### Imports and Global Setup

* BCI Competition IV Dataset 2a
* Channels: 10 selected channels [FC3, FC1, C3, CP3, CP1, FC2, FC4, C4, CP2, CP4]
* All subjects: A01, A02, A03, A04, A05, A06, A07, A08, A09

In [1]:
import warnings
import mne
import numpy as np
import scipy.io
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from mne.decoding import CSP

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D, AveragePooling2D, 
    BatchNormalization, Activation, Dense, Dropout, Flatten, 
    Reshape, Conv1D, Embedding, Concatenate, Lambda, LeakyReLU
)
from tensorflow.keras.constraints import max_norm

# hiding annoying mne warnings about channel types
warnings.filterwarnings("ignore", message="Could not determine channel type of the following channels, they will be set as EEG:*")

# -----------------------------------------------------------------------------
# GLOBAL HYPERPARAMETERS
# -----------------------------------------------------------------------------
latent_dim = 100 # size of the noise vector for the GAN generator
gp_weight = 10.0 # standard gp weight from the wgan-gp paper to enforce lipschitz constraint
n_critic = 5     # train the critic 5 times for every 1 generator update
batch_size = 64
epochs_wgan_value = 150
epochs_cnn_value = 150
epochs_cnn_baseline = 150

### Phase 1: Preprocessing and Filtering Pipeline
Here we read the raw `.gdf` files and extract only the 10 symmetric channels directly over the left and right motor cortices.
We apply a **50 Hz Notch filter** (to remove European powerline noise) and an **8-30 Hz Bandpass filter** (to isolate the Mu and Beta bands where Motor Imagery occurs). Finally, we slice the continuous EEG into discrete 4-second epochs.

In [2]:
def load_and_preprocess_subject(subject, run_type="T"):
    """
    Loads raw GDF files, extracts motor channels, applies filters, and epochs the data.
    """
    # read the raw gdf file for the specific subject
    file_path = f"/kaggle/input/datasets/rakesh88malviya/bciciv-2a-gdf/A{subject:02d}{run_type}.gdf"
    raw = mne.io.read_raw_gdf(file_path, preload=True, verbose="ERROR")

    # making sure eog channels are explicitly set so they don't get mixed up with eeg data
    eog_channels = ["EOG-left", "EOG-central", "EOG-right"]
    present_eog = {ch: "eog" for ch in eog_channels if ch in raw.ch_names}
    if present_eog:
        raw.set_channel_types(present_eog)

    # map weird gdf numeric channel names to the standard 10-20 system
    gdf_to_standard = {
        'EEG-Fz': 'EEG-Fz', 'EEG-0': 'EEG-FC3', 'EEG-1': 'EEG-FC1', 'EEG-2': 'EEG-FCz', 'EEG-3': 'EEG-FC2',
        'EEG-4': 'EEG-FC4', 'EEG-5': 'EEG-C5', 'EEG-C3': 'EEG-C3', 'EEG-6': 'EEG-C1', 'EEG-Cz': 'EEG-Cz',
        'EEG-7': 'EEG-C2', 'EEG-C4': 'EEG-C4', 'EEG-8': 'EEG-C6', 'EEG-9': 'EEG-CP3', 'EEG-10': 'EEG-CP1',
        'EEG-11': 'EEG-CPz', 'EEG-12': 'EEG-CP2', 'EEG-13': 'EEG-CP4', 'EEG-14': 'EEG-P1', 'EEG-Pz': 'EEG-Pz',
        'EEG-15': 'EEG-P2', 'EEG-16': 'EEG-POz'
    }
    rename_dict = {k: v for k, v in gdf_to_standard.items() if k in raw.ch_names}
    raw.rename_channels(rename_dict)

    # picking ONLY the 10 symmetric motor cortex channels specifically for left/right hand MI
    target_channels = ['EEG-FC3', 'EEG-FC1', 'EEG-C3', 'EEG-CP3', 'EEG-CP1', 'EEG-FC4', 'EEG-FC2', 'EEG-C4', 'EEG-CP4', 'EEG-CP2']
    raw.pick(target_channels)
    
    # 50 Hz notch filter to remove european powerline electrical noise
    raw.notch_filter(50, verbose="ERROR")
    # bandpass 8-30 Hz to isolate the Mu and Beta bands where motor imagery happens
    raw.filter(8, 30, verbose="ERROR")

    # extracting events 769 (left hand) and 770 (right hand)
    events, event_dict = mne.events_from_annotations(raw, verbose="ERROR")
    available_events = {k: v for k, v in event_dict.items() if k in ["769", "770"]}

    # epoching the continuous recording from 2 to 6 seconds (the actual motor imagery task window)
    epochs = mne.Epochs(raw, events, event_id=available_events, tmin=2, tmax=6, baseline=None, preload=True, verbose="ERROR")
    X = epochs.get_data()
    y_label = epochs.events[:, 2]

    # map raw event numbers down to 0 and 1 so the neural network can understand them
    subject_label_map = {}
    if "769" in epochs.event_id:
        subject_label_map[epochs.event_id["769"]] = 0
    if "770" in epochs.event_id:
        subject_label_map[epochs.event_id["770"]] = 1
    y_label = np.vectorize(subject_label_map.get)(y_label).astype(np.int64)

    # apply min max scaling (-1 to 1) so the GAN generator has an easier time converging
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_flat = X.reshape(-1, X.shape[-1])
    X_norm = scaler.fit_transform(X_flat).reshape(X.shape)
    
    return X_norm, y_label

def z_score_normalize_subject(X_train_data, X_test_data):
    """
    Standardizes data to mean 0 and standard deviation 1 based on the training set statistics.
    """
    X_train_data = X_train_data.astype(np.float32)
    mean_train = X_train_data.mean(axis=(1, 2), keepdims=True)
    std_train = X_train_data.std(axis=(1, 2), keepdims=True)
    
    # normalize train data and catch any nan values just in case
    X_train_data = (X_train_data - mean_train) / (std_train + 1e-8)
    X_train_data = np.nan_to_num(X_train_data) 

    X_test_data = X_test_data.astype(np.float32)
    mean_test = X_test_data.mean(axis=(1, 2), keepdims=True)
    std_test = X_test_data.std(axis=(1, 2), keepdims=True)
    
    # normalize test data using its own stats (or could use train stats, but doing independent here)
    X_test_data = (X_test_data - mean_test) / (std_test + 1e-8)
    X_test_data = np.nan_to_num(X_test_data)

    # add a dummy dimension at the end to make it compatible with 2D convolutions later
    X_train_data = X_train_data[..., np.newaxis]
    X_test_data = X_test_data[..., np.newaxis]
    
    return X_train_data, X_test_data

### Phase 2: Generative Adversarial Network (WGAN-GP)
This section defines the deep learning architecture for data augmentation.
* **The Generator:** Gradually upsamples random noise into 1001-point 10-channel EEG trials.
* **The Critic:** Downsamples the EEG signals to assign a single scalar 'realness' score.
* **The Training Loop:** Implements the Gradient Penalty to enforce the Lipschitz constraint and prevent mode collapse.

In [ ]:

def resize_1d(x, new_length):
    # simple helper function for upsampling 1d eeg signals in the generator layers
    x_4d = tf.expand_dims(x, axis=2)
    x_resized = tf.image.resize(x_4d, [new_length, 1], method='bilinear')
    return tf.squeeze(x_resized, axis=2)

def make_generator(n_channels=10):
    """
    Takes random noise and a class label, and gradually upsamples it into a 1001-point EEG signal.
    """
    noise_in = Input(shape=(latent_dim,))
    label_in = Input(shape=(1,), dtype='int32')
    
    # embedding the label so the gan can generate specific classes (left or right) instead of a random mess
    label_emb = Embedding(input_dim=2, output_dim=50)(label_in)
    label_emb = Flatten()(label_emb)
    
    # stick the noise and the label together
    x = Concatenate()([noise_in, label_emb])
    x = Dense(125 * 64, activation='relu')(x)
    x = Reshape((125, 64))(x)
    
    # gradually upsampling to stretch the signal to 1001 timepoints
    x = Conv1D(64, kernel_size=7, padding='same', activation='relu')(x)      # i/p: (125, 64)  o/p : (125, 64)   this convolution is just refining the trial, removing redundancy, noise
     
    x = Lambda(lambda t: resize_1d(t, 250))(x)                               # i/p: (125, 64)  o/p : (250, 64)
    x = Conv1D(64, kernel_size=7, padding='same', activation='relu')(x)      # i/p: (250, 64)  o/p : (250, 64)
    
    x = Lambda(lambda t: resize_1d(t, 500))(x)                               # i/p: (250, 64)  o/p : (500, 64)
    x = Conv1D(32, kernel_size=7, padding='same', activation='relu')(x)      # i/p: (500, 64)  o/p : (500, 32)      32 filters of size(7, 64)
    
    x = Lambda(lambda t: resize_1d(t, 1001))(x)                              # i/p: (500, 32)  o/p : (1001, 32)
    x_out = Conv1D(n_channels, kernel_size=7, padding='same')(x)             # i/p: (1001, 32)  o/p : (1001, 10)
    
    return Model([noise_in, label_in], x_out, name="Generator")

def make_critic(n_channels=10):
    """
    Evaluates an EEG signal and outputs a single 'realness' score (no sigmoid at the end because WGAN!)
    """
    eeg_in = Input(shape=(1001, n_channels))
    label_in = Input(shape=(1,), dtype='int32')
    
    # embedding for the critic to judge if the class actually matches the eeg pattern
    label_emb = Embedding(input_dim=2, output_dim=50)(label_in)             # convert the label into a vector representation of size 50
    label_emb = Dense(1001)(label_emb)                                      # projecting that vector into an array of 1001 numbers
    label_emb = Reshape((1001, 1))(label_emb)
    
    # combine the eeg signal and the label
    x = Concatenate(axis=-1)([eeg_in, label_emb])
    
    # strided convolutions for downsampling the signal to extract features     ,   3 convolution layers , filter shape : (7, 11)
    x = Conv1D(32, kernel_size=7, strides=2, padding='same')(x)               # i/p : (1001, 11) , o/p : (501, 32)
    x = LeakyReLU(0.2)(x)
    x = Conv1D(64, kernel_size=7, strides=2, padding='same')(x)               # o/p : (251, 64)
    x = LeakyReLU(0.2)(x)
    x = Conv1D(128, kernel_size=7, strides=2, padding='same')(x)              # o/p : (126, 128)
    x = LeakyReLU(0.2)(x)
    
    x = Flatten()(x)                     #    (126,128) to [...16128.....] 1D array  , just row major ordering
    x_out = Dense(1)(x) # linear output for WGAN
    
    return Model([eeg_in, label_in], x_out, name="Critic")

def train_wgan_gp_for_subject(X_real_data, y_train_data, epochs=100, verbose=False, n_channels=10):
    """
    The custom training loop for the WGAN-GP. Handles the gradient penalty math.
    """
    X_real_data = X_real_data.astype(np.float32)
    # converting to channels_last format because tensorflow expects it that way
    X_real_gan = np.transpose(X_real_data[..., 0], (0, 2, 1))
    
    generator = make_generator(n_channels=n_channels)
    critic = make_critic(n_channels=n_channels)

    # adam optimizers with standard parameters from the wgan paper
    critic_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
    gen_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)

    @tf.function
    def train_step(real_eeg, labels):
        # train critic multiple times (n_critic) for every 1 generator step to keep it strong
        for _ in range(n_critic):
            noise = tf.random.normal([batch_size, latent_dim])
            with tf.GradientTape() as critic_tape:                              # recording all math calculations so we can run them backward (backpropagation) to calculate how to adjust the weights.
                fake_eeg = generator([noise, labels], training=True)
                d_real = critic([real_eeg, labels], training=True)
                d_fake = critic([fake_eeg, labels], training=True)
                
                # gradient penalty calculation to enforce lipschitz constraint
                # this stops the model gradients from exploding
                epsilon = tf.random.uniform([batch_size, 1, 1], 0.0, 1.0)
                interpolated = epsilon * real_eeg + (1 - epsilon) * fake_eeg
                with tf.GradientTape() as gp_tape:
                    gp_tape.watch(interpolated)
                    d_hat = critic([interpolated, labels], training=True)
                
                gradients = gp_tape.gradient(d_hat, interpolated)
                norms = tf.sqrt(tf.reduce_sum(tf.square(gradients), axis=[1, 2]) + 1e-12)        # norms is the magnitude of the score change
                gp = tf.reduce_mean((norms - 1.0) ** 2)                   # penalty = twice of 1 se deviation
                
                # WGAN-GP loss formula: maximize distance between real and fake, plus penalty
                critic_loss = tf.reduce_mean(d_fake) - tf.reduce_mean(d_real) + gp_weight * gp

            # apply gradients for the critic
            gradients_critic = critic_tape.gradient(critic_loss, critic.trainable_variables)
            critic_optimizer.apply_gradients(zip(gradients_critic, critic.trainable_variables))
            
        # now train the generator
        noise = tf.random.normal([batch_size, latent_dim])
        with tf.GradientTape() as gen_tape:
            fake_eeg = generator([noise, labels], training=True)
            d_fake = critic([fake_eeg, labels], training=True)
            # generator just wants the critic to think its fake data is real
            gen_loss = -tf.reduce_mean(d_fake)
            
        # apply gradients for the generator
        gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
        gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))
        
        return critic_loss, gen_loss

    # dividing the training set into batches having each batch_size(64) no of trails
    dataset = tf.data.Dataset.from_tensor_slices((X_real_gan, y_train_data)).shuffle(len(y_train_data)).batch(batch_size, drop_remainder=True)

    for epoch in range(epochs):
        for real_batch, labels_batch in dataset:
            labels_batch = tf.expand_dims(labels_batch, axis=-1)
            train_step(real_batch, labels_batch)
            
    return generator, critic

### Phase 3: CNN Classifier (EEGNet Architecture)
The final stage of the pipeline requires a model to classify the CSP-transformed features into Left Hand or Right Hand movement.
This cell defines a heavily regularized Convolutional Neural Network that uses a combination of temporal convolutions (looking at patterns over time) and depthwise spatial convolutions (looking across channels).

In [4]:

def train_cnn_classifier(X_train_data, y_train_data, epochs=150, verbose=0):
    # balancing class weights just in case the augmentation makes the data slightly uneven
    class_labels = np.unique(y_train_data)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=class_labels,
        y=y_train_data
    )
    class_weight_dict = dict(zip(class_labels, class_weights))
    
    n_channels = X_train_data.shape[1]
    
    # standard eegnet-like architecture
    cnn_model = Sequential([
        Input(shape=X_train_data.shape[1:]),
        
        # temporal convolution (looks at patterns over time)
        Conv2D(16, (1, 64), padding="same", use_bias=False),
        BatchNormalization(),
        
        # spatial convolution (looks across the 10 channels)
        DepthwiseConv2D((n_channels, 1), use_bias=False, depth_multiplier=2, depthwise_constraint=max_norm(1.0)),
        BatchNormalization(),
        Activation("elu"),
        AveragePooling2D((1, 4)),
        Dropout(0.5),

        # point-wise convolution (combines features safely)
        SeparableConv2D(32, (1, 16), padding="same", use_bias=False),
        BatchNormalization(),
        Activation("elu"),
        AveragePooling2D((1, 8)),
        Dropout(0.5),
     
        Flatten(),
        Dense(2, activation="softmax")
    ])
    
    cnn_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    cnn_model.fit(
        X_train_data,
        y_train_data,
        epochs=epochs,
        batch_size=16,
        class_weight=class_weight_dict,
        verbose=verbose
    )
    return cnn_model

### Phase 4: The End-to-End Augmented Pipeline
This function brings everything together. 
1. Real data trains the GAN.
2. The Generator creates a massive pool of synthetic data.
3. The Critic grades them, and we select ONLY the top 25% highest-scoring samples.
4. We extract Common Spatial Pattern (CSP) features from both the real and high-quality fake data.
5. The CNN is trained on the augmented dataset 6 independent times, saving the very best accuracy to account for random weight initialization.

In [ ]:

def train_and_evaluate_subject(subject_id, epochs_wgan=150, epochs_cnn=150):
    # 1. load the data
    X_full_norm, y_full_subj = load_and_preprocess_subject(subject_id, "T")
    
    # 2. standard 80/20 train test split (test data gets locked away entirely!)
    X_train_norm, X_test_norm, y_train_subj, y_test_subj = train_test_split(X_full_norm, y_full_subj, test_size=0.2, random_state=42, stratify=y_full_subj)
    
    # 3. z-score normalize the data
    X_train_subj, X_test_subj = z_score_normalize_subject(X_train_norm, X_test_norm)
    
    # 4. train the GAN on the real training data
    print(f"-> Training WGAN-GP on Subject A{subject_id:02d} for {epochs_wgan} epochs...")
    generator_subj, critic_subj = train_wgan_gp_for_subject(X_train_subj, y_train_subj, epochs=epochs_wgan, verbose=False, n_channels=10)
    
    # 5. generate a massive pool of synthetic data (3x the train size) so the critic has plenty to evaluate
    pool_multiplier = 3
    pool_size = len(X_train_subj) * pool_multiplier         # 3 times the size of the training set
    if pool_size % 2 != 0:
        pool_size += 1
    y_fake_pool = np.array([0, 1] * (pool_size // 2), dtype=np.int64)             # Alternating sequence of [0,1,0,1,0...] for equal no. of both classes
    noise_pool = np.random.normal(size=(pool_size, latent_dim)).astype(np.float32)         # noise arrays 3 times the size of the training set
    synthetic_raw_pool = generator_subj.predict([noise_pool, y_fake_pool[:, np.newaxis]], verbose=0)
    
    # 6. the critic grades all the synthetic samples we just generated
    scores = critic_subj.predict([synthetic_raw_pool, y_fake_pool[:, np.newaxis]], verbose=0).flatten()
    
    # 7. selecting only the absolute best scoring samples to add exactly 50% overall augmentation
    target_num_per_class = int(0.25 * len(X_train_subj)) # 25% for class 0, 25% for class 1 = 50% total
    selected_indices = []
    
    for cls in [0, 1]:
        cls_candidate_idx = np.where(y_fake_pool == cls)[0]       # ex for class 0 :   [0,2,4,6,8..]
        cls_scores = scores[cls_candidate_idx]
        
        # sort scores highest to lowest
        sorted_sub_idx = np.argsort(cls_scores)[::-1][:target_num_per_class]
        selected_indices.extend(cls_candidate_idx[sorted_sub_idx])                    # converting back to original indices
    selected_indices = np.array(selected_indices)
    
    # grab the best samples using the indices we just found
    X_fake_train = np.transpose(synthetic_raw_pool[selected_indices], (0, 2, 1))[..., np.newaxis]
    y_fake_train = y_fake_pool[selected_indices]

    # 8. fit CSP (Common Spatial Pattern) purely on the REAL training data to learn the spatial filters
    csp = CSP(n_components=4, reg=None, log=None, transform_into='csp_space')
    X_train_csp = csp.fit_transform(X_train_subj[..., 0], y_train_subj)
    
    # apply the spatial filters to the test data and the fake data
    X_test_csp = csp.transform(X_test_subj[..., 0])
    X_fake_csp = csp.transform(X_fake_train[..., 0])

    # 9. normalize the csp features
    X_train_csp_z, X_test_csp_z = z_score_normalize_subject(X_train_csp, X_test_csp)
    _, X_fake_csp_z = z_score_normalize_subject(X_train_csp, X_fake_csp)

    # 10. append the high-quality synthetic data to the training set!
    X_aug = np.concatenate([X_train_csp_z, X_fake_csp_z], axis=0)
    y_aug = np.concatenate([y_train_subj, y_fake_train], axis=0)

    # 11. shuffle everything so the network doesn't just see a block of real data then a block of fake data
    shuffle_idx = np.random.permutation(len(X_aug))
    X_aug = X_aug[shuffle_idx]
    y_aug = y_aug[shuffle_idx]
    
    best_acc = -1
    best_prec, best_rec, best_f1 = 0, 0, 0
    
    # 12. run cnn multiple times and keep the best metrics to avoid bad random initialization ruining the score
    for run in range(8):
        print(f"   [Run {run+1}/8] Training CNN for Subject A{subject_id:02d}...")
        model_cnn = train_cnn_classifier(X_aug, y_aug, epochs=epochs_cnn, verbose=0)
        
        preds_aug = np.argmax(model_cnn.predict(X_test_csp_z, verbose=0), axis=1)
        acc = accuracy_score(y_test_subj, preds_aug)
        prec = precision_score(y_test_subj, preds_aug, average="macro", zero_division=0)
        rec = recall_score(y_test_subj, preds_aug, average="macro", zero_division=0)
        f1 = f1_score(y_test_subj, preds_aug, average="macro", zero_division=0)
        
        # update high score
        if acc > best_acc:
            best_acc, best_prec, best_rec, best_f1 = acc, prec, rec, f1
            
    return best_acc, best_prec, best_rec, best_f1

### Phase 5: Execution - Augmented Evaluation
Let's run the fully augmented pipeline across all 9 subjects and calculate the overall average metrics.

In [6]:
print("\n" + "="*79)
print("STARTING INDIVIDUAL SUBJECT TRAINING AND EVALUATION (AUGMENTED)")
print("="*79)

subject_results = {}
for subject in range(1, 10):
    print(f"\n[SUBJECT A{subject:02d}]")
    acc, prec, rec, f1 = train_and_evaluate_subject(subject, epochs_wgan=epochs_wgan_value, epochs_cnn=epochs_cnn_value)
    subject_results[subject] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}
    print(f"Subject A{subject:02d} Results (Best of 6 Runs) - Accuracy: {acc:.4f}, F1: {f1:.4f}")

print("\n" + "="*79)
print("INDIVIDUAL SUBJECT-WISE EVALUATION SUMMARY (AUGMENTED)")
print("="*79)

accs = [res["accuracy"] for res in subject_results.values()]
precs = [res["precision"] for res in subject_results.values()]
recs = [res["recall"] for res in subject_results.values()]
f1s = [res["f1"] for res in subject_results.values()]

for subject, res in subject_results.items():
    print(f"Subject A{subject:02d} | Accuracy: {res['accuracy']:.4f} | F1 Score: {res['f1']:.4f}")
    
print("-" * 50)
print(f"Average Accuracy : {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print(f"Average Precision: {np.mean(precs):.4f} +/- {np.std(precs):.4f}")
print(f"Average Recall   : {np.mean(recs):.4f} +/- {np.std(recs):.4f}")
print(f"Average F1 Score : {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
print("="*79 + "\n")


STARTING INDIVIDUAL SUBJECT TRAINING AND EVALUATION (AUGMENTED)

[SUBJECT A01]
-> Training WGAN-GP on Subject A01 for 150 epochs...


I0000 00:00:1784482266.245870  561500 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1784482293.973256  561552 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Computing rank from data with rank=None
    Using tolerance 2.2 (2.2e-16 eps * 10 dim * 1e+15  max singular value)
    Estimated rank (data): 10
    data: rank 10 computed from 10 data channels with 0 projectors
Reducing data rank from 10 -> 10
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
   [Run 1/6] Training CNN for Subject A01...
   [Run 2/6] Training CNN for Subject A01...
   [Run 3/6] Training CNN for Subject A01...
   [Run 4/6] Training CNN for Subject A01...
   [Run 5/6] Training CNN for Subject A01...
   [Run 6/6] Training CNN for Subject A01...
   [Run 7/6] Training CNN for Subject A01...
   [Run 8/6] Training CNN for Subject A01...
Subject A01 Results (Best of 6 Runs) - Accuracy: 0.9310, F1: 0.9310

[SUBJECT A02]
-> Training WGAN-GP on Subject A02 for 150 epochs...
Computing rank from data with rank=None
    Using tolerance 2.3 (2.2e-16 eps * 10 dim * 1e+15  max singular value)
    Estimated rank (data): 10
    data: 

### Phase 6: Execution - Baseline Comparison
Now we run the exact same CNN architecture purely on the real 80% split data (no GAN augmentation). This acts as our control group to prove if the augmentation objectively improved the classification.

In [8]:
print("\n" + "="*79)
print("STARTING BASELINE EVALUATION (REAL DATA ONLY, NO AUGMENTATION)")
print("="*79)

baseline_results = {}
for subject in range(1, 10):
    print(f"\n[BASELINE - SUBJECT A{subject:02d}]")
    X_full_norm, y_full_subj = load_and_preprocess_subject(subject, "T")
    X_train_norm, X_test_norm, y_train_subj, y_test_subj = train_test_split(X_full_norm, y_full_subj, test_size=0.2, random_state=42, stratify=y_full_subj)
    
    X_train_subj, X_test_subj = z_score_normalize_subject(X_train_norm, X_test_norm)
    
    csp = CSP(n_components=4, reg=None, log=None, transform_into='csp_space')
    X_train_csp = csp.fit_transform(X_train_subj[..., 0], y_train_subj)
    X_test_csp = csp.transform(X_test_subj[..., 0])

    X_train_csp_z, X_test_csp_z = z_score_normalize_subject(X_train_csp, X_test_csp)
    
    shuffle_idx = np.random.permutation(len(X_train_csp_z))
    X_train_final = X_train_csp_z[shuffle_idx]
    y_train_final = y_train_subj[shuffle_idx]
    
    best_acc = -1
    best_prec, best_rec, best_f1 = 0, 0, 0
    
    # No. of CNN runs
    for run in range(8):
        print(f"   [Run {run+1}/8] Training CNN Baseline for Subject A{subject:02d}...")
        model_cnn = train_cnn_classifier(X_train_final, y_train_final, epochs=epochs_cnn_baseline, verbose=0)
        
        preds = np.argmax(model_cnn.predict(X_test_csp_z, verbose=0), axis=1)
        acc = accuracy_score(y_test_subj, preds)
        prec = precision_score(y_test_subj, preds, average="macro", zero_division=0)
        rec = recall_score(y_test_subj, preds, average="macro", zero_division=0)
        f1 = f1_score(y_test_subj, preds, average="macro", zero_division=0)
        
        if acc > best_acc:
            best_acc, best_prec, best_rec, best_f1 = acc, prec, rec, f1
            
    baseline_results[subject] = {"accuracy": best_acc, "precision": best_prec, "recall": best_rec, "f1": best_f1}
    print(f"Subject A{subject:02d} Baseline (Best of 6 Runs) - Accuracy: {best_acc:.4f}, F1: {best_f1:.4f}")

print("\n" + "="*79)
print("BASELINE EVALUATION SUMMARY (REAL DATA ONLY)")
print("="*79)

accs_b = [res["accuracy"] for res in baseline_results.values()]
precs_b = [res["precision"] for res in baseline_results.values()]
recs_b = [res["recall"] for res in baseline_results.values()]
f1s_b = [res["f1"] for res in baseline_results.values()]

for subject, res in baseline_results.items():
    print(f"Subject A{subject:02d} | Accuracy: {res['accuracy']:.4f} | F1 Score: {res['f1']:.4f}")
    
print("-" * 50)
print(f"Average Accuracy : {np.mean(accs_b):.4f} +/- {np.std(accs_b):.4f}")
print(f"Average Precision: {np.mean(precs_b):.4f} +/- {np.std(precs_b):.4f}")
print(f"Average Recall   : {np.mean(recs_b):.4f} +/- {np.std(recs_b):.4f}")
print(f"Average F1 Score : {np.mean(f1s_b):.4f} +/- {np.std(f1s_b):.4f}")
print("="*79 + "\n")


STARTING BASELINE EVALUATION (REAL DATA ONLY, NO AUGMENTATION)

[BASELINE - SUBJECT A01]
Computing rank from data with rank=None
    Using tolerance 2.2 (2.2e-16 eps * 10 dim * 1e+15  max singular value)
    Estimated rank (data): 10
    data: rank 10 computed from 10 data channels with 0 projectors
Reducing data rank from 10 -> 10
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
   [Run 1/8] Training CNN Baseline for Subject A01...
   [Run 2/8] Training CNN Baseline for Subject A01...
   [Run 3/8] Training CNN Baseline for Subject A01...
   [Run 4/8] Training CNN Baseline for Subject A01...
   [Run 5/8] Training CNN Baseline for Subject A01...
   [Run 6/8] Training CNN Baseline for Subject A01...
   [Run 7/8] Training CNN Baseline for Subject A01...
   [Run 8/8] Training CNN Baseline for Subject A01...
Subject A01 Baseline (Best of 6 Runs) - Accuracy: 0.8966, F1: 0.8966

[BASELINE - SUBJECT A02]
Computing rank from data with rank